<a href="https://colab.research.google.com/github/MadhuriGhadge/Algerian-Forest-Fires-FWI-Prediction/blob/main/control_experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import kagglehub
path = kagglehub.dataset_download("martinthoma/hasyv2-dataset-friend-of-mnist")

100%|██████████| 105M/105M [00:01<00:00, 98.1MB/s]

Extracting files...


In [2]:
print(path)

/root/.cache/kagglehub/datasets/martinthoma/hasyv2-dataset-friend-of-mnist/versions/1


In [3]:
import os
os.listdir(path)

['HASYv2.tar.bz2',
 'symbols.csv',
 'HASYv2',
 'hasy-data-labels.csv',
 'README.txt']

In [4]:
import pandas as pd

csv_path = os.path.join(path, "hasy-data-labels.csv")
df = pd.read_csv(csv_path)

df.head()

,path,symbol_id,latex,user_id
0,hasy-data/v2-00000.png,31,A,50
1,hasy-data/v2-00001.png,31,A,10
2,hasy-data/v2-00002.png,31,A,43
3,hasy-data/v2-00003.png,31,A,43
4,hasy-data/v2-00004.png,31,A,4435


In [5]:
df['latex'].unique()[:20]

array(['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M',
       'N', 'O', 'P', 'Q', 'R', 'S', 'T'], dtype=object)

In [6]:
[df for df in df['latex'].unique() if 'times' in df or 'div' in df]

['\\times', '\\otimes', '\\div', '\\ltimes', '\\boxtimes', '\\rtimes']

In [7]:
[d for d in df['latex'].unique() if d in ['0','1','2','3','4','5','6','7','8','9']]

['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']

In [8]:
[d for d in df['latex'].unique() if d in ['+','-','=','\\times','\\div']]

['-', '+', '\\times', '\\div']

In [9]:
target_symbols = ['0','1','2','3','4','5','6','7','8','9',
                  '+','-','=','\\times','\\div']

filtered_df = df[df['latex'].isin(target_symbols)]

In [10]:
filtered_df

,path,symbol_id,latex,user_id
345,hasy-data/v2-00345.png,70,0,10
346,hasy-data/v2-00346.png,70,0,31
347,hasy-data/v2-00347.png,70,0,10
348,hasy-data/v2-00348.png,70,0,10
349,hasy-data/v2-00349.png,70,0,10
...,...,...,...,...
80483,hasy-data/v2-80483.png,526,\div,16925
80484,hasy-data/v2-80484.png,526,\div,16925
80485,hasy-data/v2-80485.png,526,\div,16925
80486,hasy-data/v2-80486.png,526,\div,16925


In [11]:
filtered_df['latex'].value_counts()

,count
latex,
\times,1509
\div,335
0,133
2,124
8,121
3,120
1,118
-,118
6,100


In [12]:
# balancing the dataset

min_samples = filtered_df['latex'].value_counts().min()

balanced_df = (
    filtered_df
    .groupby('latex')
    .apply(lambda x: x.sample(min_samples, random_state=42))
    .reset_index(drop=True)
)

balanced_df['latex'].value_counts()

/tmp/ipykernel_241/1714844873.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min_samples, random_state=42))


,count
latex,
+,61
-,61
0,61
1,61
2,61
3,61
4,61
5,61
6,61


In [13]:
'=' in df['latex'].unique()

False

In [14]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    balanced_df,
    test_size=0.2,
    stratify=balanced_df['latex'],
    random_state=42
)

print(train_df['latex'].value_counts())
print(test_df['latex'].value_counts())

latex
2         49
6         49
3         49
\div      49
9         49
\times    49
5         49
0         49
-         49
7         49
+         49
8         48
1         48
4         48
Name: count, dtype: int64
latex
8         13
4         13
1         13
-         12
2         12
+         12
\times    12
\div      12
0         12
9         12
3         12
6         12
7         12
5         12
Name: count, dtype: int64


In [15]:
# converting labels to numbers

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

train_df['label'] = le.fit_transform(train_df['latex'])
test_df['label'] = le.transform(test_df['latex'])

In [16]:
print(dict(zip(le.classes_, le.transform(le.classes_))))

{'+': np.int64(0), '-': np.int64(1), '0': np.int64(2), '1': np.int64(3), '2': np.int64(4), '3': np.int64(5), '4': np.int64(6), '5': np.int64(7), '6': np.int64(8), '7': np.int64(9), '8': np.int64(10), '9': np.int64(11), '\\div': np.int64(12), '\\times': np.int64(13)}


In [18]:
train_df['path'].head()

,path
629,hasy-data/v2-20730.png
281,hasy-data/v2-20207.png
504,hasy-data/v2-20548.png
615,hasy-data/v2-20690.png
218,hasy-data/v2-20050.png


In [21]:
def load_images(df):
    images = []

    for p in df['path']:
        img_path = os.path.join(path, "HASYv2", p)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

        if img is None:
            print("Missing:", img_path)
            continue

        img = img / 255.0
        images.append(img)

    return np.array(images)

In [22]:
X_train = load_images(train_df)
X_test = load_images(test_df)

y_train = train_df['label'].values
y_test = test_df['label'].values

In [23]:
X_train = X_train.reshape(-1, 32, 32, 1)
X_test = X_test.reshape(-1, 32, 32, 1)

In [24]:
print(X_train.shape)
print(X_test.shape)

(683, 32, 32, 1)
(171, 32, 32, 1)


In [27]:
!pip install tensorflow

In [29]:
from tensorflow.keras import layers, models

model = models.Sequential([

    layers.Input(shape=(32,32,1)),

    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),

    layers.Flatten(),

    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),

    layers.Dense(len(le.classes_), activation='softmax')
])

In [30]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [31]:
history = model.fit(
    X_train,
    y_train,
    epochs=10,
    validation_data=(X_test, y_test)
)

Epoch 1/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 8s 155ms/step - accuracy: 0.2299 - loss: 2.4873 - val_accuracy: 0.4971 - val_loss: 2.0518
Epoch 2/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5564 - loss: 1.5189 - val_accuracy: 0.6842 - val_loss: 1.0749
Epoch 3/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6764 - loss: 0.9785 - val_accuracy: 0.8012 - val_loss: 0.7679
Epoch 4/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7804 - loss: 0.7472 - val_accuracy: 0.8129 - val_loss: 0.6879
Epoch 5/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8272 - loss: 0.5241 - val_accuracy: 0.8363 - val_loss: 0.5722
Epoch 6/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8624 - loss: 0.4696 - val_accuracy: 0.8830 - val_loss: 0.5035
Epoch 7/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8975 - loss: 0.3667 - val_accuracy: 0.8830 - val_loss: 0.4735
Epoch 8/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9195 - loss: 0.2839 - val_accuracy: 0.8889 - val_lo

In [33]:
model.save("baseline_cnn_math_symbols.keras")